# Chapter 3 — Residual Networks

Chapter 2 ended with a question it could not answer: if convolutions are this
good, why not stack fifty of them? Through 2014 the field assumed you could,
and then discovered that past a certain depth **adding layers made networks
worse**, and worse on the *training* set, so not even overfitting. This chapter
diagnoses that, fixes it with a single line, and races the fix against the
disease.

| Module | What you build | Dataset (Hugging Face) |
|---|---|---|
| 1 | The vanishing-gradient problem, measured numerically in NumPy, and the identity-shortcut fix | (synthetic) |
| 2 | A real **ResNet-32** vs an identical **plain-32** network, raced head-to-head | `uoft-cs/cifar10` |

Module 2 reproduces, at CPU scale, the experiment from *Deep Residual Learning
for Image Recognition* (He et al., 2015), the most-cited paper in modern deep
learning.

It is a short chapter, deliberately. There is exactly one idea in it, the idea
is worth one line of code, and that line holds up everything in the book that
comes after.

**How each concept is presented**, the same three passes as Chapters 1–2:

> 🧠 **The intuition:** the idea in plain language, no symbols.
> 📐 **The math:** the same idea written precisely, so you can read papers.
> 💻 **The code:** the same idea again, executable, in the cell that follows.

**Runtime notes:** CIFAR-10 is already in your Hugging Face cache from
Chapter 2, so nothing new downloads. On CPU, Module 2 trains two networks at
~8–10 min each; knobs are marked `# <- knob`. Start the training cell, get
coffee.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from datasets import load_dataset

keras.utils.set_random_seed(42)
rng = np.random.default_rng(seed=42)
plt.rcParams["figure.figsize"] = (7, 4.5)

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)

---
# Module 1 — Why Depth Breaks, and the Identity Shortcut

🧠 **The intuition.** Through 2014, vision networks grew from 8 layers (AlexNet)
to 19 (VGG) and accuracy climbed. So: keep stacking? No. Past a certain depth,
**adding layers made networks *worse***, even on *training* data. Read that
again, because it rules out the obvious explanation: this is not overfitting,
it is under-*optimizing*. The deeper network could in principle copy the
shallower one and add do-nothing layers, and it fails to find that solution.
This is the **degradation problem**.

## 1.1 The gradient's journey backward

📐 **The math.** Backpropagation through a stack of layers multiplies Jacobians,
one per layer:

$$ \frac{\partial L}{\partial h_0} \;=\; J_1^\top J_2^\top \cdots J_D^\top \, \frac{\partial L}{\partial h_D} $$

A product of $D$ matrices behaves like $g^D$ where $g$ is the typical
per-layer gain. If $g < 1$ the gradient **vanishes** exponentially; if $g > 1$
it **explodes**. Hitting $g \approx 1$ exactly, for every layer, for the whole
of training, is a knife's edge.

A **residual connection** changes the layer from $h_{l+1} = f(h_l)$ to

$$ h_{l+1} = h_l + f(h_l) $$

so each Jacobian becomes $I + J_f$. The identity matrix is a **gradient
highway**: even if $J_f$ contributes almost nothing, the product contains an
unbroken chain of $I$'s and the signal reaches layer 1 intact.

💻 **The code.** Measure it rather than taking it on faith: push a gradient
backward through 100 random linear layers (per-layer gain 0.9), with and
without the shortcut.


In [ ]:
width, depth, gain = 64, 100, 0.9

# One random weight matrix per layer, scaled so its typical gain is ~0.9
Ws = [rng.normal(scale=gain / np.sqrt(width), size=(width, width)) for _ in range(depth)]

v_plain = rng.normal(size=width); v_plain /= np.linalg.norm(v_plain)   # unit gradient at the top
v_resid = v_plain.copy()

norms_plain, norms_resid = [], []
for W in Ws:
    v_plain = W.T @ v_plain              # plain layer:    J = W
    v_resid = v_resid + W.T @ v_resid    # residual layer: J = I + W  ->  J^T v = v + W^T v
    norms_plain.append(np.linalg.norm(v_plain))
    norms_resid.append(np.linalg.norm(v_resid))

plt.semilogy(norms_plain, label="plain: $h_{l+1} = W h_l$")
plt.semilogy(norms_resid, label="residual: $h_{l+1} = h_l + W h_l$")
plt.xlabel("layers travelled backward"); plt.ylabel("gradient norm (log scale)")
plt.title("The gradient highway, measured"); plt.legend(); plt.grid(True)
plt.show()

print(f"after {depth} layers -> plain: {norms_plain[-1]:.2e} | residual: {norms_resid[-1]:.2e}")

**Read the two numbers.** The plain gradient decays geometrically, roughly like
$0.9^{100} \approx 3 \times 10^{-5}$, and in this run it lands a little below
$10^{-5}$. By layer 100 it is numerically dead and early layers learn nothing.

The residual gradient does the *opposite*: it **grows**, to around $10^{12}$.
That's no accident: each factor $\|I + W\|$ is slightly *above* 1, and a
hundred of those compound too. The asymmetry is the point: the identity path
puts a **floor** under the gradient (it can never silently die), and the
*growth* side is tameable, since real ResNets keep it in check with batch
normalization and residual branches that start near zero. Vanishing has no
such fix in a plain net: once the signal is $10^{-5}$ of its original size,
no rescaling trick recovers what it should have been pointing at.

Two honest footnotes before we build the real thing:

1. **Batch normalization already fixes *outright* vanishing**, which is partly
   why VGG-19 worked at all. The degradation problem in BN networks is subtler:
   deep plain nets are simply *harder to optimize*. So in Module 2, expect the
   plain network to train *slower and worse*, not to flatline at zero.
2. There's a second, softer reason residual blocks win: each block only has to
   learn a **correction** $f(x) = H(x) - x$ to an identity default. If extra
   depth isn't useful, "do nothing" ($f \approx 0$) is trivially learnable, so
   added layers can't easily hurt. Plain layers have no such safe default:
   doing nothing means learning a full identity map through conv weights.


---
# Module 2 — ResNet vs Plain: The Race

🧠 **The experiment.** We now run the He et al. comparison at CPU scale: two
networks **identical in every way** (same depth of 32 conv layers, same
filters, same optimizer, same data, same seed), except that one has shortcuts.
Whatever difference appears is caused by the `Add()` line and nothing else.

📐 **The architecture** is the paper's actual CIFAR-10 design:

- a stem conv (16 filters), then **3 stages** of residual blocks with
  16 → 32 → 64 filters, halving resolution between stages;
- each block = Conv→BN→ReLU→Conv→BN, plus the shortcut, then a final ReLU;
- when a block changes resolution/width, the shortcut uses a 1×1 conv so the
  shapes match (you can't add a 32×32×16 tensor to a 16×16×32 one);
- global average pooling instead of a big dense head.

## 2.1 Reload CIFAR-10 (cached from Chapter 2)


In [ ]:
cifar = load_dataset("uoft-cs/cifar10")
class_names = cifar["train"].features["label"].names

def split_to_arrays(split, n=None, image_col="image", label_col="label", seed=42):
    """Shuffle a HF split, optionally take the first n rows, return (images, labels)."""
    ds = split.shuffle(seed=seed)
    if n is not None:
        ds = ds.select(range(n))
    images = np.stack([np.array(im) for im in ds[image_col]])
    labels = np.array(ds[label_col])
    return images, labels

N_TRAIN = 10_000                     # <- knob: up to 50_000
Xc_raw, yc = split_to_arrays(cifar["train"], n=N_TRAIN, image_col="img")
Xt_raw, yt = split_to_arrays(cifar["test"],  image_col="img")

Xc = (Xc_raw / 255.0).astype("float32")
Xt = (Xt_raw / 255.0).astype("float32")
print("train:", Xc.shape, "| test:", Xt.shape)

## 2.2 One block builder, one switch

The `residual` flag is the *only* difference between the two networks, a
single `Add` layer. Read the shortcut logic carefully; the 1×1 "projection"
shortcut is the detail most tutorials gloss over.


In [ ]:
def block(x, filters, downsample, residual):
    """Two 3x3 convs; optionally wrapped by an identity (or 1x1-projection) shortcut."""
    stride = 2 if downsample else 1

    y = layers.Conv2D(filters, 3, strides=stride, padding="same", use_bias=False)(x)
    y = layers.BatchNormalization()(y)
    y = layers.Activation("relu")(y)
    y = layers.Conv2D(filters, 3, padding="same", use_bias=False)(y)
    y = layers.BatchNormalization()(y)

    if residual:
        shortcut = x
        if downsample or x.shape[-1] != filters:        # shapes differ -> project with 1x1 conv
            shortcut = layers.Conv2D(filters, 1, strides=stride, use_bias=False)(x)
            shortcut = layers.BatchNormalization()(shortcut)
        y = layers.Add()([y, shortcut])                 # THE line this whole notebook is about

    return layers.Activation("relu")(y)


def build_net(residual, blocks_per_stage=5, name=None):
    """He et al.'s CIFAR-10 architecture; depth = 6*blocks_per_stage + 2."""
    inputs = keras.Input(shape=(32, 32, 3))
    x = layers.Conv2D(16, 3, padding="same", use_bias=False, name="stem")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    for stage, filters in enumerate([16, 32, 64]):
        for b in range(blocks_per_stage):
            x = block(x, filters,
                      downsample=(stage > 0 and b == 0),   # shrink 32->16->8 at stage entry
                      residual=residual)

    x = layers.GlobalAveragePooling2D()(x)                 # (8, 8, 64) -> (64,), no giant Flatten
    outputs = layers.Dense(10, activation="softmax")(x)
    return keras.Model(inputs, outputs, name=name)


BLOCKS = 5    # <- knob: depth = 6*BLOCKS + 2 = 32. Try 3 (=20, gap shrinks) or 9 (=56, gap yawns)

plain_net = build_net(residual=False, blocks_per_stage=BLOCKS, name=f"plain_{6*BLOCKS+2}")
res_net   = build_net(residual=True,  blocks_per_stage=BLOCKS, name=f"resnet_{6*BLOCKS+2}")
print(f"{plain_net.name}: {plain_net.count_params():,} params")
print(f"{res_net.name}:   {res_net.count_params():,} params")

Near-identical parameter counts (the ResNet's few extras are the 1×1 projection
shortcuts). Whatever difference we see in training is *architecture*, not capacity.

## 2.3 The race

Same compile, same epochs, same batches. On CPU each model takes roughly
8–10 minutes with the default knobs, and this is the notebook's long cell.


In [ ]:
EPOCHS = 6     # <- knob

race = {}
for model in (plain_net, res_net):
    keras.utils.set_random_seed(42)          # identical batch order + init randomness for both
    model.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    print(f"--- training {model.name} ---")
    race[model.name] = model.fit(Xc, yc,
                                 validation_split=0.1,
                                 epochs=EPOCHS,
                                 batch_size=128,
                                 verbose=2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, h in race.items():
    style = dict(marker="o", linestyle="--" if name.startswith("plain") else "-")
    axes[0].plot(h.history["accuracy"], label=f"{name} (train)", **style)
    axes[1].plot(h.history["val_accuracy"], label=name, **style)
axes[0].set_title("training accuracy"); axes[1].set_title("validation accuracy")
for ax in axes:
    ax.set_xlabel("epoch"); ax.legend(); ax.grid(True)
plt.tight_layout()
plt.show()

for model in (plain_net, res_net):
    loss, acc = model.evaluate(Xt, yt, verbose=0)
    print(f"{model.name:>10} test accuracy: {acc:.4f}")

**What you should see:** the ResNet's *training* accuracy pulls ahead early and
stays ahead. With identical capacity, it is simply easier to optimize. That's
the degradation problem, and its cure, in one plot.

The gap scales with depth, exactly as Module 1 predicts. In the paper's full
version of this experiment, plain-56 was *worse than plain-20*, while ResNet-56
kept improving, and ResNet-152 won ImageNet 2015. Set `BLOCKS = 9` (depth 56)
sometime when you can spare the CPU time and watch the plain net fall apart.

Every modern architecture (ResNets, U-Nets, and every Transformer, including
the ones behind ChatGPT and Claude) is built out of residual blocks. The
`Add()` line above is arguably the most important single line in deep learning.


---
# Wrap-Up

| You built | The transferable lesson |
|---|---|
| NumPy gradient-highway experiment | Products of Jacobians vanish/explode; $I + J_f$ guarantees a path back |
| ResNet-32 vs plain-32 race | Depth needs shortcuts; identical capacity, radically different trainability |

**Where this leads:** residual connections are what hold the **Transformer** up.
Every attention block and every MLP block in GPT-style models sits inside
exactly the $x + f(x)$ pattern you built in Module 2, for exactly the reasons
you measured in Module 1. You will build that pattern again, from a completely
different starting point, and recognise it immediately.

Nearer term: you now have two ways to train a vision model well, and both of
them assume you have the data. The next chapter is what to do when you don't.
